# Cycle 3 — Preprocessing: Player Injury Risk

**Project:** Football Predictor  
**Depends on:** `cycle3_exploration_injuries.ipynb`  
**Output:** `data/processed/player_injuries_processed.csv`

---

## Steps
1. Load raw data
2. Drop post-season leakage columns
3. Drop non-informative identifier columns
4. Construct binary target (High_Injury: season_days_injured >= 28)
5. Fill first-season NaN with 0 (no prior history)
6. Impute pace/physic with median
7. Fill position_numeric/work_rate_numeric with mode
8. Final validation and save

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../../data/raw/player_injuries.csv')
print(f'Raw shape: {df.shape}')

Raw shape: (1301, 30)


## Step 1 — Construct Target Variable

In [2]:
THRESHOLD = 28
df['High_Injury'] = (df['season_days_injured'] >= THRESHOLD).astype(int)
counts = df['High_Injury'].value_counts()
print(f'Target: High_Injury (season_days_injured >= {THRESHOLD} days)')
print(f'  Low Injury  (0): {counts[0]} ({counts[0]/len(df)*100:.1f}%)')
print(f'  High Injury (1): {counts[1]} ({counts[1]/len(df)*100:.1f}%)')

Target: High_Injury (season_days_injured >= 28 days)
  Low Injury  (0): 388 (29.8%)
  High Injury (1): 913 (70.2%)


## Step 2 — Drop Post-Season Leakage and Identifier Columns

In [3]:
drop_cols = [
    # Identifiers
    'p_id2', 'dob', 'nationality', 'work_rate', 'position', 'start_year',
    # Post-season leakage
    'season_days_injured', 'total_days_injured',
    'season_minutes_played', 'season_games_played', 'season_matches_in_squad',
    'total_minutes_played', 'total_games_played',
]
df = df.drop(columns=drop_cols)
print(f'After dropping leakage/identifiers: {df.shape}')
print(f'Remaining columns: {list(df.columns)}')

After dropping leakage/identifiers: (1301, 18)
Remaining columns: ['height_cm', 'weight_kg', 'pace', 'physic', 'fifa_rating', 'age', 'cumulative_minutes_played', 'cumulative_games_played', 'minutes_per_game_prev_seasons', 'avg_days_injured_prev_seasons', 'avg_games_per_season_prev_seasons', 'bmi', 'work_rate_numeric', 'position_numeric', 'significant_injury_prev_season', 'cumulative_days_injured', 'season_days_injured_prev_season', 'High_Injury']


## Step 3 — Fill First-Season NaN with 0

**Why:** Players in their first tracked season have no prior injury history. NaN means 0 history, not unknown — filling with 0 is correct.

In [4]:
history_cols = [
    'cumulative_minutes_played', 'cumulative_games_played',
    'minutes_per_game_prev_seasons', 'avg_days_injured_prev_seasons',
    'avg_games_per_season_prev_seasons', 'significant_injury_prev_season',
    'cumulative_days_injured', 'season_days_injured_prev_season'
]
before = df[history_cols].isnull().sum().sum()
df[history_cols] = df[history_cols].fillna(0)
after = df[history_cols].isnull().sum().sum()
print(f'Filled {before} NaN values in history columns with 0')
print(f'Remaining nulls in history cols: {after}')

Filled 4844 NaN values in history columns with 0
Remaining nulls in history cols: 0


## Step 4 — Impute pace, physic with Median; position_numeric with Mode

In [5]:
for col in ['pace', 'physic']:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    print(f'  {col}: filled {df[col].isnull().sum()} NaN with median={median_val:.1f}')

mode_pos = df['position_numeric'].mode()[0]
df['position_numeric'] = df['position_numeric'].fillna(mode_pos)
print(f'  position_numeric: filled NaN with mode={mode_pos}')

  pace: filled 0 NaN with median=71.0
  physic: filled 0 NaN with median=72.3
  position_numeric: filled NaN with mode=1.0


## Step 5 — Final Validation and Save

In [6]:
print('Null check:')
nulls = df.isnull().sum()
print(nulls[nulls > 0] if nulls.sum() > 0 else '  0 nulls remaining')
print()
print(f'Final shape: {df.shape}')
print(f'Features: {df.shape[1]-1} | Target: High_Injury')
print()
print(df.describe().round(2))

os.makedirs('../../data/processed', exist_ok=True)
df.to_csv('../../data/processed/player_injuries_processed.csv', index=False)
print('\nSaved: ../data/processed/player_injuries_processed.csv')

Null check:
  0 nulls remaining

Final shape: (1301, 18)
Features: 17 | Target: High_Injury

       height_cm  weight_kg     pace   physic  fifa_rating      age  \
count    1301.00    1301.00  1301.00  1301.00      1301.00  1301.00   
mean      182.52      76.83    70.37    70.95        74.51    26.64   
std         6.82       7.36    10.49     7.88         5.78     3.94   
min       163.00      58.00    28.33    40.75        53.00    17.00   
25%       178.00      72.00    64.17    67.40        71.33    24.00   
50%       183.00      76.00    71.00    72.33        75.17    27.00   
75%       187.67      82.00    77.50    76.00        78.50    29.00   
max       203.00      99.00    93.00    88.17        89.50    39.00   

       cumulative_minutes_played  cumulative_games_played  \
count                    1301.00                  1301.00   
mean                     2226.93                    28.54   
std                      4253.83                    53.28   
min                    

**Next step:** `cycle3_modelling.ipynb`